# enrich_trip - standalone debug notebook

Same logic as `pipeline/enrich_trip.py`, split into cells so each step can
be run and inspected independently. Run cells top to bottom; if something
fails, the cell that failed tells you exactly which stage is broken
(imports, DB connection, geocoding, Wikipedia, or embeddings).

## 1. Imports
If this cell fails with `ModuleNotFoundError`, the cluster/environment is missing a package - install it and re-run this cell before continuing.

In [ ]:
import requests
from sentence_transformers import SentenceTransformer

import lakebase

print("Imports OK")

## 2. Parameters
Set `trip_id` to an existing trip id from your `trips` table (the widget below creates an editable box - set it, then re-run this cell).

In [ ]:
dbutils.widgets.text("trip_id", "")
TRIP_ID = int(dbutils.widgets.get("trip_id"))
print(f"TRIP_ID = {TRIP_ID}")

## 3. Test the Lakebase connection
Confirms the secret/connection/permissions are all working, and shows which destinations this trip has before we try enriching them.

In [ ]:
destinations = lakebase.run_query(
    "SELECT id, name FROM destinations WHERE trip_id = %s", (TRIP_ID,)
)
print(f"Found {len(destinations)} destination(s) for trip {TRIP_ID}:")
for d in destinations:
    print(f"  id={d['id']}  name={d['name']}")

## 4. Load the embedding model
This downloads model weights on first run - isolated here since it's a likely slow/failure-prone step.

In [ ]:
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
WIKI_API_URL = "https://en.wikipedia.org/w/api.php"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded:", EMBEDDING_MODEL)

## 5. Helper functions
Same functions as the production script, unchanged.

In [ ]:
def geocode(name: str) -> dict | None:
    resp = requests.get(GEOCODING_URL, params={"name": name, "count": 1}, timeout=10)
    resp.raise_for_status()
    results = resp.json().get("results") or []
    if not results:
        return None
    r = results[0]
    return {
        "canonical_name": r["name"],
        "latitude": r["latitude"],
        "longitude": r["longitude"],
        "country": r.get("country"),
        "timezone": r.get("timezone"),
    }


def wikipedia_summary(title: str) -> dict | None:
    resp = requests.get(
        WIKI_API_URL,
        params={
            "action": "query",
            "prop": "extracts",
            "exintro": True,
            "explaintext": True,
            "titles": title,
            "format": "json",
        },
        timeout=10,
    )
    resp.raise_for_status()
    pages = resp.json().get("query", {}).get("pages", {})
    page = next(iter(pages.values()), None)
    if not page or "missing" in page:
        return None
    return {
        "title": page.get("title"),
        "extract": page.get("extract") or "",
        "url": f"https://en.wikipedia.org/?curid={page.get('pageid')}",
    }


def wikipedia_nearby(lat: float, lon: float, radius_m: int = 10000, limit: int = 10) -> list[dict]:
    resp = requests.get(
        WIKI_API_URL,
        params={
            "action": "query",
            "list": "geosearch",
            "gscoord": f"{lat}|{lon}",
            "gsradius": radius_m,
            "gslimit": limit,
            "format": "json",
        },
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json().get("query", {}).get("geosearch", [])


def upsert_destination_embedding(destination_id: int, text: str) -> None:
    if not text:
        return
    embedding = model.encode(text).tolist()
    lakebase.run_write(
        """
        INSERT INTO destination_embeddings (destination_id, embedding, model_name)
        VALUES (%s, %s, %s)
        ON CONFLICT (destination_id, model_name)
        DO UPDATE SET embedding = EXCLUDED.embedding, created_at = now()
        """,
        (destination_id, str(embedding), EMBEDDING_MODEL),
    )


def upsert_activity_embedding(activity_id: int, text: str) -> None:
    if not text:
        return
    embedding = model.encode(text).tolist()
    lakebase.run_write(
        """
        INSERT INTO activity_embeddings (activity_id, embedding, model_name)
        VALUES (%s, %s, %s)
        ON CONFLICT (activity_id, model_name)
        DO UPDATE SET embedding = EXCLUDED.embedding, created_at = now()
        """,
        (activity_id, str(embedding), EMBEDDING_MODEL),
    )


def enrich_destination(destination_id: int, name: str) -> None:
    geo = geocode(name)
    if geo is None:
        print(f"No geocoding match for destination {destination_id} ('{name}'), skipping.")
        return

    summary = wikipedia_summary(geo["canonical_name"]) or {}

    lakebase.run_write(
        """
        UPDATE destinations
        SET canonical_name = %s, latitude = %s, longitude = %s, country = %s,
            timezone = %s, description = %s, wikimedia_page_title = %s, wikimedia_url = %s
        WHERE id = %s
        """,
        (
            geo["canonical_name"], geo["latitude"], geo["longitude"], geo["country"],
            geo["timezone"], summary.get("extract"), summary.get("title"), summary.get("url"),
            destination_id,
        ),
    )
    upsert_destination_embedding(destination_id, summary.get("extract", ""))

    for page in wikipedia_nearby(geo["latitude"], geo["longitude"]):
        page_summary = wikipedia_summary(page["title"]) or {}
        existing = lakebase.run_query(
            "SELECT id FROM activities WHERE destination_id = %s AND wikimedia_page_title = %s",
            (destination_id, page["title"]),
        )
        if existing:
            activity_id = existing[0]["id"]
            lakebase.run_write(
                "UPDATE activities SET description = %s, latitude = %s, longitude = %s WHERE id = %s",
                (page_summary.get("extract"), page["lat"], page["lon"], activity_id),
            )
        else:
            inserted = lakebase.run_insert_returning(
                """
                INSERT INTO activities
                    (destination_id, name, description, source, latitude, longitude,
                     wikimedia_page_title, wikimedia_url)
                VALUES (%s, %s, %s, 'wikimedia', %s, %s, %s, %s)
                RETURNING id
                """,
                (
                    destination_id, page["title"], page_summary.get("extract"),
                    page["lat"], page["lon"], page["title"], page_summary.get("url"),
                ),
            )
            activity_id = inserted["id"]
        upsert_activity_embedding(activity_id, page_summary.get("extract", ""))

    print(f"Enriched destination {destination_id} ('{geo['canonical_name']}').")

print("Functions defined")

## 6. Test a single API call in isolation
Before running the full loop, sanity-check geocoding + Wikipedia against the first destination so a failure here doesn't get buried inside the loop.

In [ ]:
if destinations:
    sample = destinations[0]
    geo = geocode(sample["name"])
    print("geocode() result:", geo)
    if geo:
        print("wikipedia_summary() result:", wikipedia_summary(geo["canonical_name"]))
        print("wikipedia_nearby() result (first 3):", wikipedia_nearby(geo["latitude"], geo["longitude"])[:3])
else:
    print("No destinations to test - check TRIP_ID and that the trip has destinations.")

## 7. Run the full enrichment for this trip
Same loop as the production script.

In [ ]:
for row in destinations:
    enrich_destination(row["id"], row["name"])

print(f"Done enriching trip {TRIP_ID}.")